In [211]:
# Import libraries and engines from step 1

import pandas as pd
import numpy as np
import sys
import os

sys.path.append("../src")

from kpi_engine import kpi_rules
from anomaly_engine import detect_anomaly

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Load data 
df = pd.read_excel("../Datasets/Step 1. synthetic_warehouse_kpi_data.xlsx")

df.head()

,Week,Country,Distribution_Center,DC_Manager,Team,Team_Leader,Shift,Employee_ID,Employee_Name,SelectionRate_Cases,PickRate,ReplenishmentRate,IdleSelectionTime_pct,OnTaskTime_pct,Overtime_pct
0,WK01,Germany,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0001,Employee_0001,182.5,157.5,152.8,11.0,88.1,5.5
1,WK01,Germany,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0002,Employee_0002,183.1,151.7,139.4,8.5,81.3,2.6
2,WK01,Germany,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0003,Employee_0003,161.4,134.6,162.6,7.5,89.3,3.2
3,WK01,Germany,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0004,Employee_0004,180.6,149.2,141.5,6.8,96.4,6.0
4,WK01,Germany,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0005,Employee_0005,178.1,124.7,129.1,8.4,92.0,6.3


In [212]:

# Apply Anomaly Detection Engine
df[["Anomaly_Flag", "Anomaly_Reason"]] = df.apply(
    lambda row: pd.Series(
        detect_anomaly(
            row=row,
            kpi_rules=kpi_rules
        )
    ),
    axis=1
)

df[[
    "Week",
    "Distribution_Center",
    "DC_Manager",
    "Team",
    "Team_Leader",
    "Shift",
    "Employee_ID",
    "Anomaly_Flag",
    "Anomaly_Reason"
]].head(10)

,Week,Distribution_Center,DC_Manager,Team,Team_Leader,Shift,Employee_ID,Anomaly_Flag,Anomaly_Reason
0,WK01,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0001,Anomaly,PickRate below target (157.5 vs 160)\nIdleSelectionTime_pct above target (11.0 vs 8)\nOnTaskTime_pct below target (88.1 vs 89)
1,WK01,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0002,Anomaly,PickRate below target (151.7 vs 160)\nReplenishmentRate below target (139.4 vs 145)\nIdleSelectionTime_pct above target (8.5 vs 8)\nOnTaskTime_pct below target (81.3 vs 89)
2,WK01,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0003,Anomaly,SelectionRate_Cases below target (161.4 vs 175)\nPickRate below target (134.6 vs 160)
3,WK01,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0004,Anomaly,PickRate below target (149.2 vs 160)\nReplenishmentRate below target (141.5 vs 145)
4,WK01,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0005,Anomaly,PickRate below target (124.7 vs 160)\nReplenishmentRate below target (129.1 vs 145)\nIdleSelectionTime_pct above target (8.4 vs 8)\nOvertime_pct above target (6.3 vs 6)
5,WK01,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0006,Anomaly,SelectionRate_Cases below target (164.2 vs 175)\nPickRate below target (151.7 vs 160)\nIdleSelectionTime_pct above target (8.7 vs 8)\nOnTaskTime_pct below target (81.9 vs 89)\nOvertime_pct above target (6.6 vs 6)
6,WK01,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0007,Anomaly,ReplenishmentRate below target (134.9 vs 145)\nOvertime_pct above target (8.0 vs 6)
7,WK01,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0008,Anomaly,SelectionRate_Cases below target (157.1 vs 175)\nOvertime_pct above target (6.7 vs 6)
8,WK01,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0009,Anomaly,SelectionRate_Cases below target (174.5 vs 175)\nReplenishmentRate below target (113.6 vs 145)\nIdleSelectionTime_pct above target (9.6 vs 8)
9,WK01,FD01,Michael Weber,Team A,Leader_A,Morning,EMP0010,Anomaly,ReplenishmentRate below target (138.8 vs 145)\nOnTaskTime_pct below target (87.0 vs 89)\nOvertime_pct above target (7.8 vs 6)


In [213]:
# Identify top x / bottom x parameters from the question & Create dynamic summary

from summary_engine import extract_top_bottom_n, create_dynamic_summary

In [214]:
## Create Open AI API call function
#!pip install openai
from openai import OpenAI


In [215]:
#Open AI Key setup

#!pip install python-dotenv
from dotenv import load_dotenv

# Load .env file
load_dotenv()

key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=key)

In [216]:
###
from ai_engine import build_warehouse_prompt, ask_warehouse_copilot

In [217]:
# Generate AI response
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a professional warehouse operations analytics copilot."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

ai_answer = response.choices[0].message.content

print(ai_answer)    


**Weekly Overview of Best Employee with Highest Pick Rate Performance**

**Hierarchy Level:** Team Leader

**Top Performer:** Lukas Fischer (Team C, Evening Shift)  
- **Pick Rate:** 162.71 cases/hour  
- **Selection Rate:** 177.12 cases  
- **Replenishment Rate:** 144.71 cases  
- **Idle Selection Time Percentage:** 8.13%  
- **On Task Time Percentage:** 88.56%  
- **Overtime Percentage:** 6.09%  
- **Anomaly Count:** 117  

**Operational Insight:**  
Lukas Fischer stands out as the highest performer in terms of pick rate this week, achieving a rate of 162.71 cases/hour. His selection rate is also commendable, indicating a strong ability to select cases efficiently. The low idle selection time (8.13%) and high on-task time (88.56%) suggest effective time management and focus during his shift.

**KPI Drivers:**  
- High pick rate indicates efficiency in picking processes.
- Low idle time reflects minimal delays in operations.
- High on-task time suggests strong engagement and productiv

In [218]:
# Combine Chatbot Function + Custom Validation + OpenAI Moderation

def ask_warehouse_copilot(question, data, kpi_rules):

    try:

        # -----------------------------
        # Basic Input Validation
        # -----------------------------

        if question is None or question.strip() == "":

            return {
                "question": question,
                "moderation_status": "blocked",
                "ai_answer": "Question cannot be empty."
            }

        question_clean = question.lower().strip()

        # -----------------------------
        # Greeting Handling
        # -----------------------------

        greetings = ["hi","hello","hey","heyi","good morning","good evening","how are you"]

        if question_clean in greetings:

            return {
                "question": question,
                "moderation_status": "greeting",
                "ai_answer": (
                    "Hello! I am your GenAI Warehouse Operations Copilot. "
                    "You can ask me questions about warehouse KPI performance, "
                    "anomalies, trends, teams, managers, employees, and operational insights."
                )
            }

        # -----------------------------
        # Custom Sensitive Topic Check
        # -----------------------------

        blocked_keywords = ["password","hack","confidential","employee address","private information","personal data"]

        for keyword in blocked_keywords:

            if keyword in question_clean:

                return {
                    "question": question,
                    "moderation_status": "blocked",
                    "ai_answer": (
                        f"Question blocked because it may involve sensitive topic: {keyword}"
                    )
                }

        # -----------------------------
        # OpenAI Moderation API
        # -----------------------------

        moderation_response = client.moderations.create(model="omni-moderation-latest",input=question)

        moderation_result = moderation_response.results[0]

        if moderation_result.flagged:

            return {
                "question": question,
                "moderation_status": "blocked_by_openai_moderation",
                "ai_answer": "Question was blocked by moderation checks."
            }

        # -----------------------------
        # Project Scope Check
        # -----------------------------

        allowed_keywords = [
            "kpi",
            "pickrate",
            "pick rate",
            "selection",
            "selectionrate",
            "selection rate",
            "replenishment",
            "idle",
            "idle time",
            "ontask",
            "on task",
            "overtime",
            "anomaly",
            "anomalies",
            "dc",
            "distribution center",
            "manager",
            "dc_manager",
            "team",
            "leader",
            "team leader",
            "shift",
            "employee",
            "warehouse",
            "performance",
            "trend",
            "weekly",
            "highest",
            "lowest",
            "best",
            "worst",
            "chart",
            "graph"
        ]

        if not any(keyword in question_clean for keyword in allowed_keywords):

            return {
                "question": question,
                "moderation_status": "out_of_scope",
                "ai_answer": (
                    "This question lies outside the scope of this project. "
                    "Please ask about warehouse KPI performance, anomalies, trends, "
                    "teams, managers, shifts, employees, or operational insights."
                )
            }

        # -----------------------------
        # Dynamic KPI Aggregation
        # -----------------------------

        summary_level, context_table = create_dynamic_summary(
            data=data,
            question=question,
            kpi_rules=kpi_rules
        )

        # -----------------------------
        # Handle Empty Context
        # -----------------------------

        if context_table is None or context_table.empty:

            return {
                "question": question,
                "moderation_status": "no_data",
                "ai_answer": (
                    "This question cannot be answered from the available dataset. "
                    "Please ask a question related to the provided warehouse KPI data."
                )
            }

        # -----------------------------
        # Prompt Building
        # -----------------------------

        prompt = build_warehouse_prompt(
            question=question,
            summary_level=summary_level,
            context_table=context_table
        )

        # -----------------------------
        # OpenAI Response
        # -----------------------------

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a professional warehouse operations analytics copilot. "
                        "Answer only using the provided KPI context. "
                        "If the answer cannot be derived from the provided data, say that it lies outside the scope of the project."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2
        )

        ai_answer = response.choices[0].message.content

        return {
            "question": question,
            "moderation_status": "approved",
            "summary_level": summary_level,
            "context_table": context_table,
            "prompt": prompt,
            "ai_answer": ai_answer
        }

    except Exception as e:

        return {
            "question": question,
            "moderation_status": "error",
            "ai_answer": f"Error while processing request: {e}"
        }

In [219]:
# Evaluation Function 

def evaluate_ai_answer(question, context_table, ai_answer, kpi_rules):

    evaluation = {
        "question": question,
        "answer_length": len(ai_answer),
        "has_recommendation": any(
            word in ai_answer.lower()
            for word in ["recommend", "suggest", "action", "improve", "focus"]
        ),
        "mentions_kpi": any(
            kpi.lower() in ai_answer.lower()
            for kpi in kpi_rules.keys()
        ),
        "mentions_context_entity": False,
        "overall_status": "Needs Review"
    }

    context_columns = [
        "Distribution_Center",
        "DC_Manager",
        "Team",
        "Team_Leader",
        "Shift",
        "Employee_ID"
    ]

    for col in context_columns:

        if col in context_table.columns:

            values = context_table[col].astype(str).unique().tolist()

            for value in values[:10]:

                if value.lower() in ai_answer.lower():

                    evaluation["mentions_context_entity"] = True

    if (
        evaluation["has_recommendation"]
        and evaluation["mentions_kpi"]
        and evaluation["mentions_context_entity"]
    ):

        evaluation["overall_status"] = "Good"

    return evaluation

In [220]:
# Create Logging Function, enterprise-grade AI systems.

#What Gets Logged	Why
#user question	    understand usage
#AI answer	        debugging
#moderation status	safety monitoring
#evaluation score	AI quality tracking
#timestamps	        audit trail

from datetime import datetime

def log_ai_interaction(result, evaluation, log_file="ai_copilot_logs.csv"):

    log_entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "question": result.get("question"),
        "moderation_status": result.get("moderation_status"),
        "summary_level": result.get("summary_level"),
        "ai_answer": result.get("ai_answer"),
        "answer_length": evaluation.get("answer_length"),
        "has_recommendation": evaluation.get("has_recommendation"),
        "mentions_kpi": evaluation.get("mentions_kpi"),
        "mentions_context_entity": evaluation.get("mentions_context_entity"),
        "overall_status": evaluation.get("overall_status")
    }

    log_df = pd.DataFrame([log_entry])

    try:
        existing_logs = pd.read_csv(log_file)
        updated_logs = pd.concat([existing_logs, log_df], ignore_index=True)
    except FileNotFoundError:
        updated_logs = log_df

    updated_logs.to_csv(log_file, index=False)

    return updated_logs



In [221]:
from chart_engine import detect_chart_intent, create_chart_data, create_dynamic_chart
from summary_engine import extract_top_bottom_n

question = "Show chart comparing PickRate across DC_Manager"

chart_data, chart_intent = create_chart_data(
    data=df,
    question=question,
    kpi_rules=kpi_rules,
    extract_top_bottom_n=extract_top_bottom_n
)

print(chart_intent)

display(chart_data)

fig = create_dynamic_chart(
    chart_data=chart_data,
    chart_intent=chart_intent
)

fig.show()

{'chart_required': True, 'chart_type': 'bar', 'time_based': False, 'detected_kpi': 'PickRate'}


,DC_Manager,PickRate
4,Thomas Klein,160.27
2,Lukas Fischer,160.10
1,Julia Becker,159.62
0,Anna Schmidt,159.49
3,Michael Weber,159.33


In [222]:
#Unified AI + Chart Response Function

def warehouse_ai_assistant(question, data, kpi_rules):

    # Get AI answer

    result = ask_warehouse_copilot(
        question=question,
        data=data,
        kpi_rules=kpi_rules
    )

    # Detect chart intent

    chart_intent = detect_chart_intent(question)

    fig = None
    chart_data = None

    # Create chart only if required

    if chart_intent["chart_required"]:

        try:

            chart_data, chart_intent = create_chart_data(
                data=data,
                question=question,
                kpi_rules=kpi_rules,
                extract_top_bottom_n=extract_top_bottom_n
            )

            fig = create_dynamic_chart(
                chart_data=chart_data,
                chart_intent=chart_intent
            )

        except Exception as e:

            print(f"Chart generation error: {e}")

    return {
        "question": result.get("question"),
        "moderation_status": result.get("moderation_status"),
        "summary_level": result.get("summary_level"),
        "ai_answer": result.get("ai_answer"),
        "chart_intent": chart_intent,
        "chart_data": chart_data,
        "figure": fig
    }





In [223]:
#SHOW 

question = "Show chart comparing PickRate across DC_Manager"

assistant_result = warehouse_ai_assistant(
    question=question,
    data=df,
    kpi_rules=kpi_rules
)

print("QUESTION:")
print(assistant_result["question"])

print("\nAI ANSWER:")
print(assistant_result["ai_answer"])

print("\nCHART INTENT:")
print(assistant_result["chart_intent"])

print("\nCHART DATA:")
display(assistant_result["chart_data"])

if assistant_result["figure"] is not None:
    assistant_result["figure"].show()
else:
    print("No chart was generated.")

QUESTION:
Show chart comparing PickRate across DC_Manager

AI ANSWER:
The PickRate across DC Managers is as follows:

- Thomas Klein: 160.27
- Lukas Fischer: 160.10
- Julia Becker: 159.62
- Anna Schmidt: 159.49
- Michael Weber: 159.33

**Insights:**
- Thomas Klein has the highest PickRate at 160.27, while Michael Weber has the lowest at 159.33.
- The differences in PickRate among the managers are minimal, indicating a relatively consistent performance across the board.
- Anomaly counts are similar across all managers, ranging from 1054 to 1068, suggesting that operational anomalies are not significantly impacting PickRate performance.

This analysis is at the DC Manager level.

CHART INTENT:
{'chart_required': True, 'chart_type': 'bar', 'time_based': False, 'detected_kpi': 'PickRate'}

CHART DATA:


,DC_Manager,PickRate
4,Thomas Klein,160.27
2,Lukas Fischer,160.10
1,Julia Becker,159.62
0,Anna Schmidt,159.49
3,Michael Weber,159.33
